# 📱 Scraping Review — Honest Kartu Kredit

---

## 🎯 Tujuan
Scraping review aplikasi **Honest - Kartu Kredit** dari Google Play Store dan menyimpan hasilnya,
kemudian memfilter hanya review berbahasa Indonesia.

### App Details:
- **App Name**: Honest - Kartu Kredit
- **App ID**: `com.honestbank.android`

### Workflow:
1. 📥 Scrape Reviews dari Google Play Store
2. 💾 Simpan RAW data ke `df_honest_reviews.csv`
3. 🌍 Filter review bahasa Indonesia
4. 💾 Simpan hasil filter ke `df_honest_reviews_id.csv`

---

## 📦 Install & Import Libraries

In [19]:
%pip install -q google_play_scraper langdetect

Note: you may need to restart the kernel to use updated packages.


In [20]:
from google_play_scraper import Sort, reviews_all
from langdetect import detect, DetectorFactory
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DetectorFactory.seed = 0  # reproducibility

print("✅ Libraries berhasil diimport!")

✅ Libraries berhasil diimport!


## 🌐 Scraping Reviews dari Google Play Store

Mengambil semua review aplikasi menggunakan **google_play_scraper**
(`com.honestbank.android`, sort: `NEWEST`, lang: `id`).

> ⏱️ Proses scraping memakan waktu beberapa menit.

In [21]:
print("🚀 Memulai scraping review aplikasi Honest - Kartu Kredit...")

honest_reviews = reviews_all(
    'com.honestbank.android',
    sleep_milliseconds=0,
    lang='id',
    sort=Sort.NEWEST,
)

print(f"✅ Scraping selesai! Total review: {len(honest_reviews):,}")

🚀 Memulai scraping review aplikasi Honest - Kartu Kredit...
✅ Scraping selesai! Total review: 45,911


## 💾 Simpan RAW Data ke DataFrame & CSV

In [22]:
# Convert ke DataFrame
df_honest = pd.DataFrame(np.array(honest_reviews), columns=['content'])
df_honest = df_honest.join(pd.DataFrame(df_honest.pop('content').tolist()))

# Simpan ke CSV
RAW_CSV = 'df_honest_reviews.csv'
df_honest.to_csv(RAW_CSV, index=False)

print(f"✅ RAW data disimpan ke: {RAW_CSV}")
print(f"📊 Shape: {df_honest.shape}")
print(f"\n👁️ Preview 5 baris pertama:")
display(df_honest[['content', 'score', 'at', 'reviewCreatedVersion']].head())

✅ RAW data disimpan ke: df_honest_reviews.csv
📊 Shape: (45911, 11)

👁️ Preview 5 baris pertama:


,content,score,at,reviewCreatedVersion
0,ini adalah pertama kali saya menggunakan kartu...,5,2026-03-13 16:27:30,3.831.0
1,"pengajuan berhasil, pelayanan utk CS atas nama...",5,2026-03-13 16:22:21,3.833.1
2,Pengajuan cepat dan aman..,5,2026-03-13 16:21:04,3.833.1
3,mantap,5,2026-03-13 16:18:33,3.833.1
4,cs Ridwan terbaik,5,2026-03-13 16:16:58,3.833.1


## 🌍 Filter Review Bahasa Indonesia

Parameter `lang='id'` pada `google_play_scraper` tidak selalu menjamin 100% konten berbahasa Indonesia.
Langkah ini mendeteksi bahasa secara eksplisit dan hanya menyimpan review dengan bahasa terdeteksi `id`.

In [23]:
def detect_lang_safe(text):
    """Deteksi bahasa teks; kembalikan 'unknown' jika gagal."""
    if not isinstance(text, str) or not text.strip():
        return 'unknown'
    try:
        return detect(text)
    except Exception:
        return 'unknown'

before_count = len(df_honest)

df_honest['detected_lang'] = df_honest['content'].apply(detect_lang_safe)
df_id = df_honest[df_honest['detected_lang'] == 'id'].copy()

after_count = len(df_id)

print(f"📊 Total review sebelum filter      : {before_count:,}")
print(f"💬 Total review bahasa Indonesia    : {after_count:,}")
print(f"🗑️ Review non-ID dihapus            : {before_count - after_count:,}")

📊 Total review sebelum filter      : 45,911
💬 Total review bahasa Indonesia    : 39,324
🗑️ Review non-ID dihapus            : 6,587


## 💾 Simpan Hasil Filter ke CSV

In [24]:
ID_CSV = 'df_honest_reviews_id.csv'
df_id.to_csv(ID_CSV, index=False)

print(f"✅ Data bahasa Indonesia disimpan ke: {ID_CSV}")
print(f"📊 Shape: {df_id.shape}")
print(f"\n👁️ Preview 5 baris pertama:")
display(df_id[['content', 'score', 'at', 'detected_lang']].head())

✅ Data bahasa Indonesia disimpan ke: df_honest_reviews_id.csv
📊 Shape: (39324, 12)

👁️ Preview 5 baris pertama:


,content,score,at,detected_lang
0,ini adalah pertama kali saya menggunakan kartu...,5,2026-03-13 16:27:30,id
1,"pengajuan berhasil, pelayanan utk CS atas nama...",5,2026-03-13 16:22:21,id
2,Pengajuan cepat dan aman..,5,2026-03-13 16:21:04,id
4,cs Ridwan terbaik,5,2026-03-13 16:16:58,id
5,penjelasan yang baik oleh mba rima,5,2026-03-13 16:11:32,id
